[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-02-reliable-tool-calls-validation-retries-idempotency.ipynb)

# Advanced Discussion 2 — validation, selective retries, idempotency keys and circuit breakers

Pydantic argument validation, retry policies, duplicate-refund simulation with and without idempotency keys, and a circuit breaker.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in seconds.

In [ ]:
%pip install -q pydantic

In [ ]:
import json, time, numpy as np
from pydantic import BaseModel, Field, ValidationError
rng = np.random.RandomState(0)

## 1. validate tool arguments before running anything

In [ ]:
class Refund(BaseModel):
    order_id: str = Field(pattern=r"^ORD-\d{6}$")
    amount: float = Field(gt=0, le=500)
    reason: str = Field(min_length=3, max_length=200)
good = '{"order_id": "ORD-123456", "amount": 42.5, "reason": "damaged item"}'
bad_examples = ['{"order_id": "123456", "amount": 42.5, "reason": "damaged"}', '{"order_id": "ORD-123456", "amount": -5, "reason": "damaged"}',
                '{"order_id": "ORD-123456", "amount": 9000, "reason": "damaged"}', '{"order_id": "ORD-123456", "amount": "abc", "reason": "x"}', '{"order_id": "ORD-123456"}']
print("valid call parses:", Refund.model_validate_json(good).amount)
for b in bad_examples:
    try: Refund.model_validate_json(b); print("  accepted (!):", b)
    except ValidationError as e:
        err = e.errors()[0]; print("  rejected: %-16s %s" % (".".join(map(str, err["loc"])), err["msg"]))

## 2. transient failures: retry policies compared (simulated tool that fails 30% of calls, 10 ms of 'work' per call)

In [ ]:
def call_tool(fail_p, r): return r.rand() >= fail_p
def policy_run(policy, fail_p=0.30, n_tasks=20000, max_attempts=4, seed=0):
    r = np.random.RandomState(seed); ok = 0; attempts = 0; waited = 0.0
    for _ in range(n_tasks):
        for a in range(max_attempts):
            attempts += 1
            if call_tool(fail_p, r): ok += 1; break
            if policy == "none": break
            if policy == "fixed": waited += 0.0
            if policy == "backoff": waited += min(8.0, 0.5 * 2 ** a) * (0.5 + r.rand() * 0.5)          # jittered exponential backoff (seconds)
    return ok / n_tasks, attempts / n_tasks, waited / n_tasks
print("\ntool failing 30% of calls, up to 4 attempts:")
print("policy               success   attempts/task   mean wait/task (s)")
for pol in ("none", "fixed", "backoff"):
    s_, a_, w_ = policy_run(pol); print("%-19s  %.4f    %.3f           %.2f" % (pol + (" (immediate)" if pol == "fixed" else ""), s_, a_, w_))
print("theory: 1 - 0.3**4 = %.4f" % (1 - 0.3 ** 4))
print("but if failures are PERSISTENT (a wrong argument, a revoked key) retries only add load:")
s_, a_, _ = policy_run("fixed", fail_p=1.0, n_tasks=2000); print("  100%% failing tool: success %.2f, attempts/task %.1f (4x the calls, no benefit)" % (s_, a_))

## 3. timeouts that hide success: at-least-once delivery and idempotency keys

In [ ]:
class PaymentService:
    def __init__(self, use_keys): self.ledger = []; self.use_keys = use_keys; self.seen = {}
    def refund(self, order, amount, key, r, timeout_after_apply=0.15):
        if self.use_keys and key in self.seen: return self.seen[key]                       # replay: return the recorded result, apply nothing
        self.ledger.append((order, amount)); result = "refunded"
        if self.use_keys: self.seen[key] = result
        if r.rand() < timeout_after_apply: raise TimeoutError("response lost")             # the refund WAS applied, but the caller never hears
        return result
def simulate(use_keys, n=5000, seed=1):
    r = np.random.RandomState(seed); svc = PaymentService(use_keys)
    for i in range(n):
        key = f"task-{i}"
        for attempt in range(3):
            try: svc.refund(f"ORD-{i}", 10.0, key, r); break
            except TimeoutError: continue                                                  # a naive agent retries after a timeout
    per_order = {}
    for o, _ in svc.ledger: per_order[o] = per_order.get(o, 0) + 1
    return sum(1 for c in per_order.values() if c > 1), len(per_order)
for use_keys in (False, True):
    dup, total = simulate(use_keys); print("idempotency keys %-5s: %d of %d orders refunded more than once (%.1f%%)" % (use_keys, dup, total, 100 * dup / total))

## 4. circuit breaker: stop hammering a dependency that is down

In [ ]:
class Breaker:
    def __init__(self, threshold=5, cooldown=20): self.fails = 0; self.open_until = -1; self.threshold = threshold; self.cooldown = cooldown
    def allow(self, t): return t >= self.open_until
    def record(self, ok, t):
        if ok: self.fails = 0
        else:
            self.fails += 1
            if self.fails >= self.threshold: self.open_until = t + self.cooldown; self.fails = 0
def outage(use_breaker, ticks=400, down=(100, 250)):
    r = np.random.RandomState(2); br = Breaker(); calls = 0; served = 0; fast_failed = 0
    for t in range(ticks):
        if use_breaker and not br.allow(t): fast_failed += 1; continue
        calls += 1; up = not (down[0] <= t < down[1]); ok = up and r.rand() > 0.02; served += ok
        if use_breaker: br.record(ok, t)
    return calls, served, fast_failed
for ub in (False, True):
    c, s_, f_ = outage(ub); print("breaker %-5s: calls made to the failing service %3d, successful %3d, failed instantly (no call) %3d" % (ub, c, s_, f_))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.